In [65]:
import pandas as pd
import numpy as np

In [66]:
data_path='../raw_datasets/'
eth_regs=pd.read_csv(f'{data_path}eth_domains_registrations.csv').drop(['registrant_address','registration_date'], axis=1)
eth_regs=eth_regs.rename(columns={'labelhash_hex':'labelhash'})

In [67]:
eth_to_usd=pd.read_csv(f'{data_path}ETHUSDT_1d.csv')
eth_to_usd['open_time_ms'] = pd.to_datetime(eth_to_usd['open_time_ms'], unit='ms')
eth_price=eth_to_usd.sort_values(by='open_time_ms', ascending=False).iloc[0]['open_price']
eth_regs['eth_reg_price']=np.round(eth_regs['price']*eth_price,2)
eth_regs = eth_regs.drop(columns=['price'])

Переходим к историям транзакций

In [68]:
eth_sales_base=pd.read_csv(f'{data_path}eth_domains_sales_base.csv').drop(['tx_hash','unix_timestamp'], axis=1)
eth_sales_wrap=pd.read_csv(f'{data_path}eth_domains_sales_wrapped.csv').drop(['tx_hash','unix_timestamp'], axis=1)


У врап версии есть значения price_usd = "nil" 

In [69]:
eth_sales_wrap['price_usd'].value_counts()

price_usd
0.207501    51
0.208007    49
0.149491    43
0.190588    42
0.261588    38
            ..
0.052750     1
0.050442     1
0.042153     1
0.039923     1
0.042753     1
Name: count, Length: 24253, dtype: int64

In [70]:
eth_sales_wrap=eth_sales_wrap[eth_sales_wrap['price_usd']!='<nil>']
eth_sales_wrap['price_usd']=eth_sales_wrap['price_usd'].astype(float)
eth_sales_base['namehash'] = eth_sales_base['labelhash'].map(eth_regs.set_index('labelhash')['namehash'])


In [71]:
eth_sales_base['namehash'].isna().sum()

np.int64(196643)

16 пропущенных значений в namehash, эти строки нам не нужны т.к это те домены которых нет в списке регистраций

Объединяем списки транзакций

In [72]:
eth_sales=pd.concat([eth_sales_wrap,
eth_sales_base.drop('labelhash', axis=1)], ignore_index=True
)

In [73]:
eth_sales_agg= (
    eth_sales.groupby('namehash', as_index=False).agg(
        eth_sales_count=('price_usd', 'count'),
        eth_sales_volume_usd=('price_usd', 'sum'),
        eth_last_sale_price_usd=('price_usd', 'last'),
    )
)

In [74]:
eth_res=eth_regs.merge(eth_sales_agg, on='namehash', how='left')


Откидываем eth из domain_name

In [75]:
eth_res['domain_name']=eth_res['domain_name'].apply( lambda x : x.split(".")[0] )

То же самое с sol 

In [76]:
sol_reg=pd.read_csv(f'{data_path}sol_domains_registrations.csv').drop(['tx_signature','bidder_key','unix_timestamp'], axis=1)
sol_sales=pd.read_csv(f'{data_path}sol_domains_sales.csv').drop(['tx_signature','bidder_key','unix_timestamp'], axis=1)

In [77]:
sol_sales_agg = (
    sol_sales.groupby('domain_key', as_index=False)
    .agg(
        sol_sales_count=('usd_price', 'count'),
        sol_sales_volume_usd=('usd_price', 'sum'),
        sol_last_sale_price_usd=('usd_price', 'last'),
    )
)

In [78]:
sol_res=sol_reg.merge(sol_sales_agg, on='domain_key', how='left')
sol_res=sol_res.rename(columns={"usd_price": "sol_reg_price"})

In [79]:
eth_res.drop(['namehash','labelhash'], axis=1,inplace=True)
sol_res.drop('domain_key', axis=1,inplace=True)
result_df=eth_res.merge(sol_res, on='domain_name', how='left')

In [80]:
len(result_df[result_df.notnull().all(axis=1)])

2461

In [81]:
result_df['domain_name'].duplicated().sum()

np.int64(0)

In [82]:
result_df.to_csv('../datasets/secondary_prices.csv', index=False)